# 11 Scaling Enterprise RAG & Security

**Real-World Scenario**: Multi-Tenant Enterprise HR Portal with Strict Role-Based Security (ACLs).

This notebook demonstrates **Role-Based ACL Metadata Payload Filtering** in LangChain, saving vector indexes into a hidden `.vectordb/` folder, completed by an **End-to-End GenAI Security LLM Response Call**.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Multi-Tenant Role Payload Filtering & Vector Indexing

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

docs = [
    "Public Employee Handbook: All staff receive 20 days paid vacation.",
    "Manager Salary Budget: Senior manager salary band is $180k-$220k.",
    "Executive Acquisition Plan: Project Titan acquisition budget $50M."
]
metadatas = [{"role": "Public"}, {"role": "Manager"}, {"role": "Executive"}]

if os.getenv("OPENAI_API_KEY"):
    vectorstore = FAISS.from_texts(docs, OpenAIEmbeddings(), metadatas=metadatas)
    save_path = os.path.join(".vectordb", "11_acl_index")
    vectorstore.save_local(save_path)
    print(f"=== Multi-Tenant ACL Index Saved to Hidden Folder: {save_path} ===")


## Step 3: Executing Complete GenAI Multi-Tenant Security LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    save_path = os.path.join(".vectordb", "11_acl_index")
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.load_local(save_path, embeddings, allow_dangerous_deserialization=True)
    
    # User with Manager role querying vector store
    matched_docs = vectorstore.similarity_search("Manager Salary Band", k=1, filter={"role": "Manager"})
    context_text = matched_docs[0].page_content
    
    prompt = ChatPromptTemplate.from_template("Answer salary query strictly using authorized ACL context:\nContext:\n{context}\n\nQuery: {query}\nAnswer:")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(context=context_text, query="What is manager salary band?"))
    print("=== Complete Multi-Tenant ACL GenAI Response ===")
    print(response.content)
